In [ ]:
CSV_FILE = "/data/philei/tt-metal/generated/profiler/reports/2026_02_24_15_34_58/ops_perf_results_2026_02_24_15_34_58.csv"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact


In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv(CSV_FILE)

# Rename the columns for clarity
df = df.rename(columns={
    'OP CODE': 'Operation',
    'HOST DURATION [ns]': 'Host Time',
    'OP TO OP LATENCY [ns]': 'Time Between Ops',
    'DEVICE FW DURATION [ns]': 'Device Time'
})


# Filter out rows before compilation finished
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('compilation_finished', na=False)
matching_indices = df.index[mask]
assert not matching_indices.empty, "No 'compilation_finished' found in ProfilerNoopOperation attributes"
latest_compilation_flag = matching_indices[-1]
df = df.iloc[latest_compilation_flag + 1:]

# Find number of training steps
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('iteration_', na=False)
matching_indices = df.index[mask]
num_training_steps = 3 # for backward compatibility, will be removed before merge
if not matching_indices.empty:
    filtered_df = df[df.index.isin(matching_indices)]
    num_training_steps = len(filtered_df['ATTRIBUTES'].unique())

df = df[df['Operation'] != 'ProfilerNoopOperation']

all_operations = df['Operation'].unique()

num_training_steps

In [ ]:
def draw_diagrams_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']
    topk = 15

    for col in time_columns:
        # ----- top-k + “Others” slice -------------------------------------
        sorted_times = grouped[col].sort_values(ascending=False)
        top_times    = sorted_times.head(topk)
        others_sum   = sorted_times.iloc[topk:].sum()
        if others_sum:
            top_times = pd.concat([top_times, pd.Series({'Others': others_sum})])

        labels = top_times.index.tolist()
        sizes  = top_times.values
        pct    = 100 * sizes / sizes.sum()

        # ----- plot -------------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 8))
        wedges, _ = ax.pie(sizes, startangle=140)   # no autopct ⇒ nothing on pie

        legend_text = [f'{lbl} — {p:.1f}%' for lbl, p in zip(labels, pct)]
        ax.legend(
            wedges,
            legend_text,
            title=f'Operations (share of {aggregation})',
            loc='center left',
            bbox_to_anchor=(1, 0.5)
        )

        ax.set_title(f'Top {topk} Operations by {aggregation} {col}')
        ax.axis('equal')
        plt.show()

        # display(fig)    # ➋ show it once

In [ ]:
draw_diagrams_with_aggregation('sum')

In [ ]:
draw_diagrams_with_aggregation('mean')

In [ ]:
def name_per_aggregation(aggregation):
    if aggregation == 'sum':
        return 'Total (per training step)'
    elif aggregation == 'mean':
        return 'Average'
    else:
        raise ValueError(f"Unsupported aggregation: {aggregation}")
    
def draw_charts_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']

    # Loop through each time column to create horizontal bar charts and pie charts
    for col in time_columns:
        # Extract the total times per operation for the current column
        total_times = grouped[col] / 1_000_000  # Convert from nanoseconds to milliseconds
        if aggregation == 'sum':
            total_times /= num_training_steps  # Normalize by number of training steps if aggregation is 'sum'
        
        # Create a horizontal bar chart
        plt.figure(figsize=(10, 6))
        total_times.sort_values().plot(kind='barh', color='skyblue') 
        plt.title(f'{name_per_aggregation(aggregation)} {col} per Operation')
        plt.xlabel(f'{name_per_aggregation(aggregation)} {col} (ms)')
        plt.ylabel('Operation')
        plt.tight_layout()
        plt.show()

In [ ]:
draw_charts_with_aggregation('sum')

In [ ]:
draw_charts_with_aggregation('mean')

In [ ]:
# @interact(operation=all_operations)
# def draw_per_operation_stats(operation):
#     df_op = df[df['Operation'] == operation]
#     if df_op.empty:
#         print(f"No data available for operation: {operation}")
#         return

#     metrics = [
#         ('Host Time (ms)',          df_op['Host Time']        / 1_000_000),
#         ('Time Between Ops (ms)',   df_op['Time Between Ops'] / 1_000_000),
#         ('Device Time (ms)',        df_op['Device Time']      / 1_000_000),
#     ]

#     for title, series in metrics:
#         # --- build figure explicitly so we know which one to close ------------
#         fig, ax = plt.subplots(figsize=(10, 6))
#         ax.plot(series.index, series.values, marker='o', color='red')
#         ax.set_title(f'{operation} – {title}')
#         ax.set_xlabel('Index')
#         ax.set_ylabel(title)
#         ax.grid(True)
#         fig.tight_layout()
#         plt.show()

In [ ]:
# example how to manually extract performance data for a specific operation
e = df[df['Operation'] == 'Untilize']
e = e[['Operation', 'Host Time', 'Time Between Ops', 'Device Time']]
e

In [ ]:
metrics = ['Host Time', 'Time Between Ops', 'Device Time']

def anomaly_detection_per_operation(operation, metric):
    df_op = df[df['Operation'] == operation]
    if df_op.empty:
        print(f"No data available for operation: {operation}")
        return

    # Calculate the mean and standard deviation for each metric
    
    series = df_op[metric]
    
    mean = series.mean()
    std_dev = series.std()
    
    # Identify anomalies as points that are more than 3 standard deviations from the mean
    anomalies = series[(series < mean - 3 * std_dev) | (series > mean + 3 * std_dev)]
    
    if not anomalies.empty:
        return df_op.loc[anomalies.index]
    return None

In [ ]:
import json

def is_not_nan(value):
    """Check if a value is not NaN."""
    return pd.notna(value) and value != 'NaN' and value != 'nan' and value != ''

# @interact(operation=all_operations, metric_name=metrics)
# def show_anomalies_attributes_per_metric(operation, metric_name):
#     anomaly_df = anomaly_detection_per_operation(operation, metric_name)

#     if anomaly_df is None:
#         print(f"No anomalies detected for {operation} in {metric_name}.")
#         return

#     for index, row in anomaly_df.iterrows():
#         attr = row['ATTRIBUTES']
#         core_count = row['CORE COUNT']
#         metric_value = row[metric_name]

#         # find all columns with prefix `INPUT_`
#         input_columns = [col for col in row.index if col.startswith('INPUT_')]
#         input_values = {col: row[col] for col in input_columns if is_not_nan(row[col])}

#         # improve print of dictionary with json
#         input_values = json.dumps(input_values, indent=8)

#         if isinstance(attr, str):
#             attr = attr.replace(';', ',')
#             attr = attr.replace('\'', '"')
#             try:
#                 attr = json.loads(attr)
#             except:
#                 pass
#             attr = json.dumps(attr, indent=8) if isinstance(attr, dict) else attr

#         print(f"Anomaly at index {index}: ")
#         print(f"    {metric_name} = {metric_value / 1_000_000} ms")
#         print(f"    core count = {core_count}")
#         print(f"    inputs = {input_values}")
#         print(f"    attributes = {attr}")


In [ ]:
# ── Training Phase Timing Analysis from Profiler Markers ────────────────
# Uses extract_results.py for parsing, then visualizes.

import sys
from pathlib import Path

# Infer tt-train path from notebook location: .../tt-train/tools/profiling/
_nb_dir = Path.cwd() if "__vsc_ipynb_file__" not in dir() else Path(__vsc_ipynb_file__).parent
TT_TRAIN = (_nb_dir / ".." / "..").resolve()
if str(TT_TRAIN / "tools") not in sys.path:
    sys.path.insert(0, str(TT_TRAIN / "tools"))

from profiling.results_json.extract_results import extract_timings, print_timings, PHASE_COLS

timings = extract_timings(Path(CSV_FILE))
assert timings is not None, "No timing data found in CSV"

# Print per-device tables
print_timings(timings)

# ── Visualize (first device) ──
first_dev_key = next(iter(timings))
dev_data = timings[first_dev_key]
iters = dev_data["iterations"]
avail_phases = [c for c in PHASE_COLS if any(it.get(c, 0) > 0 for it in iters)]
phase_labels = [c.replace("_ms", "").replace("_", " ").title() for c in avail_phases]

iters_df = pd.DataFrame(iters)
first_iter = iters_df["iteration"].min()
steady = iters_df[iters_df["iteration"] > first_iter]
avg = steady[avail_phases].mean()

if not steady.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0']

    ax = axes[0]
    avg.plot(kind='bar', ax=ax, color=colors[:len(avail_phases)])
    ax.set_xticklabels(phase_labels, rotation=30, ha='right')
    ax.set_title(f'{first_dev_key}: Average Phase Duration (excl. iter {first_iter})')
    ax.set_ylabel('Time (ms)')
    for i_bar, v in enumerate(avg):
        ax.text(i_bar, v + avg.max() * 0.01, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

    ax = axes[1]
    for ci, col in enumerate(avail_phases):
        ax.plot(steady['iteration'], steady[col], marker='o',
                label=phase_labels[ci], color=colors[ci % len(colors)])
    ax.set_title(f'{first_dev_key}: Phase Duration per Iteration (excl. iter {first_iter})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Time (ms)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Op-by-op execution timeline for one iteration ──
# Shows each op in execution order with training phase marker
# boundaries and total device time per segment.

TIMELINE_DEVICE_ID = 0       # which device to plot
TIMELINE_ITERATION = 3       # which iteration (post-compilation)
X_TICK_INTERVAL_MS = 0.5     # x-axis tick spacing in ms
MARKERS_PER_CALL = 10        # marker rows per profiler call (must match extract_results.py)
SAVE_PLOT = True             # True: save to file instead of showing inline
SAVE_FORMAT = "svg"          # "png" or "svg"
SAVE_DPI = 200               # DPI for png (ignored for svg)

import re as _re
import numpy as _np
from pathlib import Path
from profiling.results_json.extract_results import PHASE_MAP

raw_tl = pd.read_csv(CSV_FILE, low_memory=False)
raw_tl['DEVICE KERNEL DURATION [ns]'] = pd.to_numeric(
    raw_tl['DEVICE KERNEL DURATION [ns]'], errors='coerce'
)

# Strip everything up to compilation_finished
comp_mask = (raw_tl['OP CODE'] == 'ProfilerNoopOperation') & \
            raw_tl['ATTRIBUTES'].str.contains('compilation_finished', na=False)
comp_indices = raw_tl.index[comp_mask]
assert not comp_indices.empty, "No compilation_finished marker found"
raw_tl = raw_tl.iloc[comp_indices[-1] + 1:].reset_index(drop=True)

# Tag marker identifiers
raw_tl['_ident'] = None
noop = raw_tl['OP CODE'] == 'ProfilerNoopOperation'
raw_tl.loc[noop, '_ident'] = raw_tl.loc[noop, 'ATTRIBUTES'].str.extract(
    r"'identifier':\s*'([^']+)'", expand=False
)

iter_start = f'iteration_{TIMELINE_ITERATION}'
iter_end = f'iteration_{TIMELINE_ITERATION + 1}'
s_idx = raw_tl.index[raw_tl['_ident'] == iter_start]
e_idx = raw_tl.index[raw_tl['_ident'] == iter_end]
assert not s_idx.empty, f"Iteration {TIMELINE_ITERATION} start marker not found"

s = s_idx[0]
e = e_idx[0] if not e_idx.empty else len(raw_tl)

it_df = raw_tl.iloc[s:e].copy().reset_index(drop=True)
is_marker = it_df['_ident'].notna()

# Group consecutive marker rows (MARKERS_PER_CALL per profiler call)
mrows = it_df[is_marker].reset_index()
marker_groups = []
mi = 0
while mi + MARKERS_PER_CALL <= len(mrows):
    ident = mrows.iloc[mi]['_ident']
    grp = mrows.iloc[mi:mi + MARKERS_PER_CALL]
    if (grp['_ident'] == ident).all():
        marker_groups.append({'ident': ident, 'last_pos': int(grp['index'].iloc[-1])})
        mi += MARKERS_PER_CALL
    else:
        mi += 1

# Non-marker ops for the target device
ops = it_df[(it_df['DEVICE ID'] == TIMELINE_DEVICE_ID) & (~is_marker)].copy()
ops = ops.reset_index().rename(columns={'index': 'iter_pos'}).reset_index(drop=True)
assert not ops.empty, f"No ops for device {TIMELINE_DEVICE_ID} in iteration {TIMELINE_ITERATION}"

ops['duration_ms'] = (ops['DEVICE KERNEL DURATION [ns]'] / 1e6).fillna(0)

# Map each marker to a y-position among the plotted ops
marker_positions = []
for mg in marker_groups:
    y = int((ops['iter_pos'] <= mg['last_pos']).sum()) - 0.5
    marker_positions.append({'ident': mg['ident'], 'y': y})

# Build phase segments between consecutive markers
segments = []
for si in range(len(marker_groups) - 1):
    a_pos, b_pos = marker_groups[si]['last_pos'], marker_groups[si + 1]['last_pos']
    seg_ops = ops[(ops['iter_pos'] > a_pos) & (ops['iter_pos'] < b_pos)]

    a_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si]['ident'])
    b_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si + 1]['ident'])
    phase = PHASE_MAP.get((a_norm, b_norm), f"{a_norm} -> {b_norm}")
    label = phase.replace('_ms', '').replace('_', ' ').title() if phase.endswith('_ms') else phase

    segments.append({
        'phase': phase, 'label': label,
        'total_ms': seg_ops['duration_ms'].sum(), 'n_ops': len(seg_ops),
        'y_start': marker_positions[si]['y'], 'y_end': marker_positions[si + 1]['y'],
    })

# Remaining ops after last marker
if marker_groups:
    rem = ops[ops['iter_pos'] > marker_groups[-1]['last_pos']]
    if not rem.empty:
        segments.append({
            'phase': 'remaining', 'label': 'Remaining',
            'total_ms': rem['duration_ms'].sum(), 'n_ops': len(rem),
            'y_start': marker_positions[-1]['y'], 'y_end': len(ops) - 0.5,
        })

# ── Plot ──
fig_height = max(6, len(ops) * 0.12)
fig, ax = plt.subplots(figsize=(18, fig_height))

y_pos = range(len(ops))
q95 = ops['duration_ms'].quantile(0.95)
bar_colors = ['#F44336' if d > q95 else '#2196F3' for d in ops['duration_ms']]

ax.barh(y_pos, ops['duration_ms'], color=bar_colors, height=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(ops['OP CODE'], fontsize=6)
ax.invert_yaxis()

x_max = ops['duration_ms'].max()
ax.set_xticks(_np.arange(0, x_max + X_TICK_INTERVAL_MS, X_TICK_INTERVAL_MS))
ax.set_xlabel('Duration (ms)')
ax.set_title(
    f'Device {TIMELINE_DEVICE_ID}, Iteration {TIMELINE_ITERATION}: '
    f'Op Execution Timeline ({len(ops)} ops, total={ops["duration_ms"].sum():.1f} ms)'
)
ax.grid(True, axis='x', alpha=0.3)

# Phase background bands
band_colors = ['#BBDEFB', '#FFE0B2', '#C8E6C9', '#FFCDD2', '#E1BEE7', '#FFF9C4']
for si, seg in enumerate(segments):
    ax.axhspan(seg['y_start'], seg['y_end'], alpha=0.15,
               color=band_colors[si % len(band_colors)], zorder=0)

# Marker lines + right-side labels
for mp in marker_positions:
    ax.axhline(y=mp['y'], color='#D32F2F', linewidth=2, linestyle='--', zorder=5)
    ax.annotate(
        f"  {mp['ident']}",
        xy=(1.01, mp['y']), xycoords=ax.get_yaxis_transform(),
        fontsize=8, fontweight='bold', color='#D32F2F',
        va='center', ha='left', annotation_clip=False,
    )

# Segment phase labels with device time
for si, seg in enumerate(segments):
    mid_y = (seg['y_start'] + seg['y_end']) / 2
    ax.annotate(
        f"{seg['label']}\n{seg['total_ms']:.2f} ms  ({seg['n_ops']} ops)",
        xy=(1.14, mid_y), xycoords=ax.get_yaxis_transform(),
        fontsize=7, va='center', ha='left', annotation_clip=False,
        bbox=dict(boxstyle='round,pad=0.3',
                  facecolor=band_colors[si % len(band_colors)], alpha=0.5, edgecolor='none'),
    )

fig.tight_layout()
fig.subplots_adjust(right=0.68)

if SAVE_PLOT:
    out_dir = Path(CSV_FILE).parent
    out_name = f"op_timeline_dev{TIMELINE_DEVICE_ID}_iter{TIMELINE_ITERATION}.{SAVE_FORMAT}"
    out_path = out_dir / out_name
    save_kwargs = {'bbox_inches': 'tight'}
    if SAVE_FORMAT == 'png':
        save_kwargs['dpi'] = SAVE_DPI
    fig.savefig(out_path, **save_kwargs)
    plt.close(fig)
    print(f"Saved {fig_height:.0f}×18 inch plot ({len(ops)} ops) → {out_path}")
else:
    plt.show()

# Summary table
print(f"\n{'Phase':<25s} {'Device Time (ms)':>16s} {'Ops':>6s}")
print("\u2500" * 50)
for seg in segments:
    print(f"{seg['label']:<25s} {seg['total_ms']:>16.2f} {seg['n_ops']:>6d}")
print("\u2500" * 50)
total_ms = sum(s['total_ms'] for s in segments)
total_ops = sum(s['n_ops'] for s in segments)
print(f"{'Total':<25s} {total_ms:>16.2f} {total_ops:>6d}")

print(f"\nTop 10 slowest ops:")
top = ops.nlargest(10, 'duration_ms')[['OP CODE', 'duration_ms']].reset_index(drop=True)
top.index += 1
display(top)


In [ ]:
# ── Zone summary: all marker segments with device time and op counts ──
# Reuses raw_tl, marker detection, and ops from the previous cell.

SUMMARY_DEVICE_ID = TIMELINE_DEVICE_ID
SUMMARY_ITERATION = TIMELINE_ITERATION

import re as _re
from profiling.results_json.extract_results import PHASE_MAP

# Build a flat list of all ops (non-marker) with their zone assignment
zone_records = []
for si in range(len(marker_groups) - 1):
    a_pos, b_pos = marker_groups[si]['last_pos'], marker_groups[si + 1]['last_pos']
    seg_ops = ops[(ops['iter_pos'] > a_pos) & (ops['iter_pos'] < b_pos)].copy()

    a_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si]['ident'])
    b_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si + 1]['ident'])
    phase = PHASE_MAP.get((a_norm, b_norm), f"{marker_groups[si]['ident']} -> {marker_groups[si + 1]['ident']}")

    seg_ops['zone'] = phase
    seg_ops['zone_order'] = si
    zone_records.append(seg_ops)

# Remaining ops after last marker
if marker_groups:
    rem = ops[ops['iter_pos'] > marker_groups[-1]['last_pos']].copy()
    if not rem.empty:
        rem['zone'] = 'remaining'
        rem['zone_order'] = len(marker_groups) - 1
        zone_records.append(rem)

zone_df = pd.concat(zone_records, ignore_index=True)

# ── 1. Per-occurrence zone summary (every segment individually) ──
zone_summary = (
    zone_df.groupby(['zone_order', 'zone'])
    .agg(
        total_device_ms=('duration_ms', 'sum'),
        num_ops=('duration_ms', 'count'),
        mean_op_ms=('duration_ms', 'mean'),
        max_op_ms=('duration_ms', 'max'),
    )
    .reset_index()
    .sort_values('zone_order')
    .drop(columns='zone_order')
    .reset_index(drop=True)
)

# ── 2. Aggregated zone summary (repeated zones merged) ──
zone_agg = (
    zone_df.groupby('zone')
    .agg(
        total_device_ms=('duration_ms', 'sum'),
        num_ops=('duration_ms', 'count'),
        mean_op_ms=('duration_ms', 'mean'),
        max_op_ms=('duration_ms', 'max'),
    )
    .reset_index()
    .sort_values('total_device_ms', ascending=False)
    .reset_index(drop=True)
)
zone_agg['occurrences'] = zone_agg['zone'].map(zone_summary['zone'].value_counts())
zone_agg['pct'] = 100.0 * zone_agg['total_device_ms'] / zone_agg['total_device_ms'].sum()

# ── 3. Aggregated per-zone per-op breakdown ──
zone_op_agg = (
    zone_df.groupby(['zone', 'OP CODE'])
    .agg(
        total_ms=('duration_ms', 'sum'),
        count=('duration_ms', 'count'),
        mean_ms=('duration_ms', 'mean'),
        max_ms=('duration_ms', 'max'),
    )
    .reset_index()
    .sort_values(['zone', 'total_ms'], ascending=[True, False])
    .reset_index(drop=True)
)

# ── 4. Per-occurrence per-zone per-op breakdown ──
zone_op_summary = (
    zone_df.groupby(['zone_order', 'zone', 'OP CODE'])
    .agg(
        total_ms=('duration_ms', 'sum'),
        count=('duration_ms', 'count'),
        mean_ms=('duration_ms', 'mean'),
        max_ms=('duration_ms', 'max'),
    )
    .reset_index()
    .sort_values(['zone_order', 'total_ms'], ascending=[True, False])
    .drop(columns='zone_order')
    .reset_index(drop=True)
)

print(f"Device {SUMMARY_DEVICE_ID}, Iteration {SUMMARY_ITERATION}")

print(f"\n{'='*80}")
print("Aggregated Zone Summary (repeated zones merged, sorted by total time)")
print(f"{'='*80}")
display(zone_agg.style.format({
    'total_device_ms': '{:.2f}',
    'mean_op_ms': '{:.4f}',
    'max_op_ms': '{:.4f}',
    'pct': '{:.1f}%',
}).hide(axis='index'))

print(f"\n{'='*80}")
print("Aggregated Per-Zone Op Breakdown (repeated zones merged)")
print(f"{'='*80}")
display(zone_op_agg.style.format({
    'total_ms': '{:.4f}',
    'mean_ms': '{:.4f}',
    'max_ms': '{:.4f}',
}).hide(axis='index'))

print(f"\n{'='*80}")
print("Per-Occurrence Zone Summary (every segment individually)")
print(f"{'='*80}")
display(zone_summary.style.format({
    'total_device_ms': '{:.2f}',
    'mean_op_ms': '{:.4f}',
    'max_op_ms': '{:.4f}',
}).hide(axis='index'))

print(f"\n{'='*80}")
print("Per-Occurrence Op Breakdown (sorted by total device time within each zone)")
print(f"{'='*80}")
display(zone_op_summary.style.format({
    'total_ms': '{:.4f}',
    'mean_ms': '{:.4f}',
    'max_ms': '{:.4f}',
}).hide(axis='index'))
